# Cleaning BRFSS 2015 for Diabetes-Risk Modelling

The raw file `2015.csv` is the CDC **Behavioral Risk Factor Surveillance System
(BRFSS) 2015** survey: **441,456 respondents × 330 columns** of survey codes.

**Goal of this notebook** — turn that raw survey into a small, tidy, analysis-ready
table for predicting diabetes risk:

1. **Target** — recode `DIABETE3` into three *stages*:
   `0 = no diabetes`, `1 = pre-diabetes / borderline`, `2 = diabetes`.
2. **Cofactors** — keep only the risk factors known to matter for diabetes /
   chronic disease (blood pressure, cholesterol, smoking, obesity/BMI, age, sex,
   race, diet, exercise, alcohol, income, marital status, checkup recency,
   education, healthcare coverage, mental health, …) and **drop the other ~300
   columns**.
3. **Recode** every kept column from its BRFSS numeric code into a clean,
   human-meaningful value, dropping *Don't know / Refused / Missing* responses.
4. **Save** the result to `diabetes_2015_clean.csv`.

> Every recode below follows the official *BRFSS 2015 Codebook*. Calculated
> variables (the `_`-prefixed ones such as `_RFHYPE5`, `_BMI5`, `_AGEG5YR`) are
> CDC's own cleaned/derived fields, so we prefer them over the raw questions
> where one exists.

## 1. Setup & load only the columns we need

The raw CSV is ~540 MB. Loading all 330 columns wastes memory, so we read only
the ~26 source columns this cleaning touches via `usecols`.

In [1]:
import numpy as np
import pandas as pd

RAW_PATH   = '../data/raw/2015.csv'
CLEAN_PATH = '../data/processed/diabetes_2015_clean.csv'

# Raw BRFSS columns we will use as SOURCES for the clean features
SRC_COLS = [
    'DIABETE3',                                   # target
    '_RFHYPE5', 'TOLDHI2', '_CHOLCHK',            # BP / cholesterol
    '_BMI5', '_BMI5CAT',                          # obesity / BMI
    'SMOKE100', '_SMOKER3',                       # smoking
    'CVDSTRK3', '_MICHD',                         # comorbid cardiovascular
    'CHCKIDNY', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2',# other chronic comorbidities
    '_TOTINDA', '_PACAT1', '_FRTLT1', '_VEGLT1',  # exercise / diet
    '_RFDRHV5',                                   # heavy alcohol
    'HLTHPLN1', 'MEDCOST',                        # healthcare access
    'GENHLTH', 'MENTHLTH', 'PHYSHLTH', 'DIFFWALK',# general / mental / physical health
    'SEX', '_AGEG5YR', '_AGE80', '_RACE', 'MARITAL',# demographics
    'EDUCA', 'INCOME2', 'EMPLOY1', 'CHECKUP1',    # SES / checkup recency
]

raw = pd.read_csv(RAW_PATH, usecols=SRC_COLS)
print(f'Raw subset loaded: {raw.shape[0]:,} rows × {raw.shape[1]} source columns')
raw.head()

Raw subset loaded: 441,456 rows × 34 source columns


,GENHLTH,PHYSHLTH,MENTHLTH,HLTHPLN1,MEDCOST,CHECKUP1,TOLDHI2,CVDSTRK3,CHCCOPD1,HAVARTH3,...,_AGEG5YR,_AGE80,_BMI5,_BMI5CAT,_SMOKER3,_RFDRHV5,_FRTLT1,_VEGLT1,_TOTINDA,_PACAT1
0,5.0,15.0,18.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,...,9.0,63.0,4018.0,4.0,3.0,1.0,2.0,1.0,2.0,4.0
1,3.0,88.0,88.0,2.0,1.0,4.0,2.0,2.0,2.0,2.0,...,7.0,52.0,2509.0,3.0,1.0,1.0,2.0,2.0,1.0,2.0
2,4.0,15.0,88.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,...,11.0,71.0,2204.0,2.0,9.0,9.0,9.0,9.0,9.0,9.0
3,5.0,30.0,30.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,...,9.0,63.0,2819.0,3.0,4.0,1.0,1.0,2.0,2.0,4.0
4,5.0,20.0,88.0,1.0,2.0,1.0,2.0,2.0,2.0,1.0,...,9.0,61.0,2437.0,2.0,4.0,1.0,9.0,1.0,2.0,4.0


## 2. The target — diabetes *stage* from `DIABETE3`

`DIABETE3` asks *"(Ever told) you have diabetes?"*. Its codes:

| code | meaning | â†’ our stage |
|------|---------|-------------|
| 1 | Yes | **2 — Diabetes** |
| 2 | Yes, but only during pregnancy | 0 — No diabetes* |
| 3 | No | **0 — No diabetes** |
| 4 | No, pre-diabetes or borderline | **1 — Pre-diabetes** |
| 7 / 9 | Don't know / Refused | drop |

\*Gestational-only diabetes is treated as "no diabetes" for chronic-risk purposes,
which is the standard convention for this dataset.

In [2]:
stage_map = {1: 2, 2: 0, 3: 0, 4: 1}          # 7, 9, NaN -> dropped below
df = raw.copy()
df['DiabetesStage'] = df['DIABETE3'].map(stage_map)

before = len(df)
df = df.dropna(subset=['DiabetesStage'])
df['DiabetesStage'] = df['DiabetesStage'].astype(int)
print(f'Dropped {before - len(df):,} rows with unusable diabetes status '
      f'(Don\'t know / Refused / Missing)')

labels = {0: 'No diabetes', 1: 'Pre-diabetes', 2: 'Diabetes'}
dist = df['DiabetesStage'].value_counts().sort_index()
for k, n in dist.items():
    print(f'  {k} {labels[k]:<13}: {n:>7,}  ({n/len(df):5.1%})')

Dropped 798 rows with unusable diabetes status (Don't know / Refused / Missing)
  0 No diabetes  : 375,712  (85.3%)
  1 Pre-diabetes :   7,690  ( 1.7%)
  2 Diabetes     :  57,256  (13.0%)


## 3. Risk-factor â†’ BRFSS variable map

Every requested risk factor is matched to the best available BRFSS 2015 column.
Binary flags are coded **1 = yes / has-risk, 0 = no** unless noted.

| Risk factor | Clean column | Source | Encoding |
|---|---|---|---|
| High blood pressure | `HighBP` | `_RFHYPE5` | 1=yes,0=no |
| High cholesterol | `HighChol` | `TOLDHI2` | 1=yes,0=no |
| Cholesterol checked (â‰¤5y) | `CholCheck` | `_CHOLCHK` | 1=yes,0=no |
| BMI | `BMI` | `_BMI5` | continuous |
| Obesity band | `BMICategory` | `_BMI5CAT` | 1=under,2=normal,3=over,4=obese |
| Smoking (â‰¥100 cigs) | `Smoker` | `SMOKE100` | 1=yes,0=no |
| Smoking status | `SmokerStatus` | `_SMOKER3` | 0=never,1=former,2=some days,3=daily |
| Stroke history | `Stroke` | `CVDSTRK3` | 1=yes,0=no |
| Heart disease / MI | `HeartDiseaseorAttack` | `_MICHD` | 1=yes,0=no |
| Kidney disease | `KidneyDisease` | `CHCKIDNY` | 1=yes,0=no |
| COPD | `COPD` | `CHCCOPD1` | 1=yes,0=no |
| Arthritis | `Arthritis` | `HAVARTH3` | 1=yes,0=no |
| Depression (ever told) | `Depression` | `ADDEPEV2` | 1=yes,0=no |
| Exercise (any, 30d) | `PhysActivity` | `_TOTINDA` | 1=yes,0=no |
| Physical-activity level | `PhysActivityLevel` | `_PACAT1` | 1=highly active…4=inactive |
| Diet – fruit ≥1/day | `Fruits` | `_FRTLT1` | 1=yes,0=no |
| Diet – veg ≥1/day | `Veggies` | `_VEGLT1` | 1=yes,0=no |
| Heavy alcohol use | `HvyAlcoholConsump` | `_RFDRHV5` | 1=yes,0=no |
| Healthcare coverage | `AnyHealthcare` | `HLTHPLN1` | 1=yes,0=no |
| Skipped doctor (cost) | `NoDocbcCost` | `MEDCOST` | 1=yes,0=no |
| General health | `GenHlth` | `GENHLTH` | 1=excellent…5=poor |
| Mental health (bad days/30) | `MentHlth` | `MENTHLTH` | 0–30 |
| Physical health (bad days/30) | `PhysHlth` | `PHYSHLTH` | 0–30 |
| Difficulty walking | `DiffWalk` | `DIFFWALK` | 1=yes,0=no |
| Sex | `Sex` | `SEX` | 1=male,0=female |
| Age band | `Age` | `_AGEG5YR` | 1–13 (5-yr bands) |
| Age in years | `AgeYears` | `_AGE80` | 18–80 continuous |
| Race / ethnicity | `Race` | `_RACE` | 1–8 category |
| Marital status | `MaritalStatus` | `MARITAL` | 1–6 category |
| Education | `Education` | `EDUCA` | 1–6 ordinal |
| Household income | `Income` | `INCOME2` | 1–8 ordinal |
| Employment status | `Employment` | `EMPLOY1` | 1–8 category |
| Time since last checkup | `CheckupTime` | `CHECKUP1` | 1=≤1y…4=5y+, 5=never |

**Not available in BRFSS 2015 core:** *Sleep* — there is no general sleep-duration
question in the 2015 file (the only sleep fields belong to optional asthma /
depression modules asked of small sub-samples), so sleep is **excluded** rather
than filled with mostly-missing data.

## 4. Recode every cofactor

We build the clean columns from the raw codes. `pd.Series.map` leaves any value
not in the mapping as `NaN`, which conveniently turns every *Don't know / Refused*
code into a missing value that we drop in the next step.

In [3]:
yes1_no2 = {1: 1, 2: 0}      # BRFSS 'Yes=1 / No=2' questions -> 1/0
rf_no1_yes2 = {1: 0, 2: 1}   # BRFSS calculated risk flags 'No=1 / Yes=2' -> 0/1

clean = pd.DataFrame({'DiabetesStage': df['DiabetesStage']})

# --- Blood pressure & cholesterol ------------------------------------------
clean['HighBP']    = df['_RFHYPE5'].map(rf_no1_yes2)
clean['HighChol']  = df['TOLDHI2'].map(yes1_no2)
clean['CholCheck'] = df['_CHOLCHK'].map({1: 1, 2: 0, 3: 0})   # 3 = never checked

# --- Obesity / BMI ----------------------------------------------------------
clean['BMI']         = df['_BMI5'] / 100.0                     # _BMI5 is BMI × 100
clean['BMICategory'] = df['_BMI5CAT']                          # 1..4, NaN dropped later

# --- Smoking & cardiovascular comorbidities --------------------------------
clean['Smoker']               = df['SMOKE100'].map(yes1_no2)
clean['SmokerStatus']         = df['_SMOKER3'].map({4: 0, 3: 1, 2: 2, 1: 3})  # never->daily; 9 drop
clean['Stroke']               = df['CVDSTRK3'].map(yes1_no2)
clean['HeartDiseaseorAttack'] = df['_MICHD'].map(yes1_no2)

# --- Other chronic comorbidities that co-occur with diabetes ---------------
clean['KidneyDisease'] = df['CHCKIDNY'].map(yes1_no2)
clean['COPD']          = df['CHCCOPD1'].map(yes1_no2)
clean['Arthritis']     = df['HAVARTH3'].map(yes1_no2)
clean['Depression']    = df['ADDEPEV2'].map(yes1_no2)

# --- Exercise & diet --------------------------------------------------------
clean['PhysActivity']      = df['_TOTINDA'].map(yes1_no2)
clean['PhysActivityLevel'] = df['_PACAT1'].where(df['_PACAT1'].between(1, 4))  # 9 = missing
clean['Fruits']            = df['_FRTLT1'].map(yes1_no2)
clean['Veggies']           = df['_VEGLT1'].map(yes1_no2)

# --- Alcohol & healthcare access -------------------------------------------
clean['HvyAlcoholConsump'] = df['_RFDRHV5'].map(rf_no1_yes2)
clean['AnyHealthcare']     = df['HLTHPLN1'].map(yes1_no2)
clean['NoDocbcCost']       = df['MEDCOST'].map(yes1_no2)

# --- General / mental / physical health ------------------------------------
clean['GenHlth']  = df['GENHLTH'].where(df['GENHLTH'].between(1, 5))
clean['MentHlth'] = df['MENTHLTH'].replace(88, 0).where(df['MENTHLTH'].isin(list(range(1, 31)) + [88]))
clean['PhysHlth'] = df['PHYSHLTH'].replace(88, 0).where(df['PHYSHLTH'].isin(list(range(1, 31)) + [88]))
clean['DiffWalk'] = df['DIFFWALK'].map(yes1_no2)

# --- Demographics -----------------------------------------------------------
clean['Sex']           = df['SEX'].map({1: 1, 2: 0})           # 1 male / 0 female
clean['Age']           = df['_AGEG5YR'].where(df['_AGEG5YR'].between(1, 13))  # 14 = missing
clean['AgeYears']      = df['_AGE80']                          # imputed age 18-80, no missing
clean['Race']          = df['_RACE'].where(df['_RACE'].between(1, 8))         # 9  = missing
clean['MaritalStatus'] = df['MARITAL'].where(df['MARITAL'].between(1, 6))     # 9  = refused

# --- SES & checkup recency --------------------------------------------------
clean['Education']   = df['EDUCA'].where(df['EDUCA'].between(1, 6))           # 9  = refused
clean['Income']      = df['INCOME2'].where(df['INCOME2'].between(1, 8))       # 77,99 dropped
clean['Employment']  = df['EMPLOY1'].where(df['EMPLOY1'].between(1, 8))       # 9  = refused
clean['CheckupTime'] = df['CHECKUP1'].map({1: 1, 2: 2, 3: 3, 4: 4, 8: 5})    # 8 never->5; 7,9 drop

print(f'Built {clean.shape[1]-1} clean cofactors + target')
clean.head()

Built 33 clean cofactors + target


,DiabetesStage,HighBP,HighChol,CholCheck,BMI,BMICategory,Smoker,SmokerStatus,Stroke,HeartDiseaseorAttack,...,DiffWalk,Sex,Age,AgeYears,Race,MaritalStatus,Education,Income,Employment,CheckupTime
0,0,1.0,1.0,1.0,40.18,4.0,1.0,1.0,0.0,0.0,...,1.0,0,9.0,63.0,1.0,1.0,4.0,3.0,8.0,1.0
1,0,0.0,0.0,0.0,25.09,3.0,1.0,3.0,0.0,0.0,...,0.0,0,7.0,52.0,1.0,2.0,6.0,1.0,3.0,4.0
2,0,0.0,1.0,1.0,22.04,2.0,NaN,NaN,1.0,NaN,...,NaN,0,11.0,71.0,1.0,2.0,4.0,NaN,7.0,1.0
3,0,1.0,1.0,1.0,28.19,3.0,0.0,0.0,0.0,0.0,...,1.0,0,9.0,63.0,1.0,1.0,4.0,8.0,8.0,1.0
4,0,0.0,0.0,1.0,24.37,2.0,0.0,0.0,0.0,0.0,...,0.0,0,9.0,61.0,1.0,1.0,5.0,NaN,8.0,1.0


## 5. Drop rows with any missing / refused value

Because every unusable code became `NaN` above, a single `dropna` removes all the
*Don't know / Refused / Missing* responses at once. (For a modelling notebook you
might instead impute these, but the brief here is a **clean** testing dataset, so
we keep only complete cases.)

In [4]:
before = len(clean)
clean = clean.dropna().reset_index(drop=True)

# Everything except continuous BMI is integer-valued -> tidy up the dtypes
int_cols = [c for c in clean.columns if c != 'BMI']
clean[int_cols] = clean[int_cols].astype(int)

print(f'Complete-case rows kept: {len(clean):,}  (dropped {before - len(clean):,}, '
      f'{(before-len(clean))/before:.1%} of the diabetes-labelled sample)')
print(f'Final shape: {clean.shape[0]:,} rows × {clean.shape[1]} columns')

Complete-case rows kept: 240,731  (dropped 199,927, 45.4% of the diabetes-labelled sample)
Final shape: 240,731 rows × 34 columns


## 6. Sanity checks & save

Quick look at the cleaned distributions and a face-validity check: diabetes
prevalence should rise steeply with BMI band, age, and high blood pressure.

In [5]:
# Class balance of the target
print('Target — DiabetesStage:')
for k, n in clean['DiabetesStage'].value_counts().sort_index().items():
    print(f'  {k} {labels[k]:<13}: {n:>7,}  ({n/len(clean):5.1%})')

# Face validity: share with diabetes (stage 2) by a few key risk factors
print('\nDiabetes prevalence (stage==2) by risk factor — should trend up:')
diab = (clean['DiabetesStage'] == 2)
for col in ['BMICategory', 'HighBP', 'Age']:
    print(f'\n  by {col}:')
    print((diab.groupby(clean[col]).mean() * 100).round(1).to_string())

Target — DiabetesStage:
  0 No diabetes  : 203,180  (84.4%)
  1 Pre-diabetes :   4,324  ( 1.8%)
  2 Diabetes     :  33,227  (13.8%)

Diabetes prevalence (stage==2) by risk factor — should trend up:

  by BMICategory:
BMICategory
1     5.4
2     5.9
3    11.8
4    24.0

  by HighBP:
HighBP
0     6.0
1    24.3

  by Age:
Age
1      1.3
2      1.8
3      2.9
4      4.5
5      6.5
6      8.7
7     11.7
8     13.7
9     17.1
10    20.1
11    21.8
12    21.2
13    18.4


In [6]:
clean.to_csv(CLEAN_PATH, index=False)
print(f'Saved cleaned dataset -> {CLEAN_PATH}')
print(f'{clean.shape[0]:,} rows × {clean.shape[1]} columns')
print('\nColumns:', ', '.join(clean.columns))

Saved cleaned dataset -> diabetes_2015_clean.csv
240,731 rows × 34 columns

Columns: DiabetesStage, HighBP, HighChol, CholCheck, BMI, BMICategory, Smoker, SmokerStatus, Stroke, HeartDiseaseorAttack, KidneyDisease, COPD, Arthritis, Depression, PhysActivity, PhysActivityLevel, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, AgeYears, Race, MaritalStatus, Education, Income, Employment, CheckupTime


## Notes & decisions

* **Target choice.** `DIABETE3` is the only diabetes-status question in BRFSS, so
  the three stages (no / pre-diabetes / diabetes) come directly from it. Pre-diabetes
  is a small class (~1.8%) — for a binary model just collapse stage 1 into 0 or 2.
* **Added comorbid cofactors.** Beyond the classic set we include conditions that
  strongly co-occur with / raise the risk of type-2 diabetes: `KidneyDisease`,
  `COPD`, `Arthritis`, `Depression`, plus richer `SmokerStatus`,
  `PhysActivityLevel`, `Employment` and a continuous `AgeYears`. (Kidney/heart
  disease can be both risk factor *and* consequence of diabetes — treat them as
  correlates, not proven causes.)
* **`_`-prefixed sources** (`_RFHYPE5`, `_BMI5`, `_MICHD`, `_TOTINDA`, `_PACAT1`,
  `_FRTLT1`, `_VEGLT1`, `_RFDRHV5`, `_SMOKER3`, `_AGEG5YR`, `_AGE80`, `_RACE`,
  `_BMI5CAT`) are CDC's pre-derived variables — already edited for skip patterns
  and out-of-range values.
* **Deliberately excluded to avoid leakage**: `DIABAGE2` (age at diabetes dx),
  `PDIABTST`, `PREDIAB1`, `INSULIN`, `BLDSUGAR` — these are only asked *because*
  someone already has (pre-)diabetes, so they would leak the target.
* **`_DRNKWEK` (drinks/week) skipped**: alcohol is already captured by
  `HvyAlcoholConsump`, and its implied-decimal scaling is easy to misread.
* **Dropped ~295 columns**: survey administration, weighting, and disease modules
  unrelated to diabetes risk (cancer screening, caregiving, HIV testing, etc.).
* **Complete-case only**: we drop rather than impute. Swap the `dropna` in §5 for a
  `SimpleImputer` inside a modelling pipeline if you prefer to retain more rows.
* **Sleep excluded**: no general sleep-duration item exists in BRFSS 2015 core.